# Nemotron **v32** — forum-correct SFT + weighted loss + expert-key repackage (submittable)

Built on three forum findings (Mark Susol, NemotronH + Unsloth):
1. **`out_proj` LoRA is DEAD** on Unsloth (fused Mamba scan output lacks `requires_grad` →
   checkpoint backward guard zeroes its gradient). `x_proj`/`dt_proj` are fused into `in_proj`.
   → **drop them**; only `in_proj` is a live Mamba LoRA target.
2. **`prepare_model_for_training` silently freezes** any LoRA param whose name isn't `.lora_A.`/
   `.lora_B.` dotted → custom expert params can train at 0. → **requires_grad audit** re-enables them.
3. **The Kaggle evaluator exposes 128 per-expert `nn.Linear`** (`experts.{j}.up_proj`), but Unsloth
   trains routed experts **FUSED** → the saved fused keys **don't match** the evaluator → **856M of
   trained expert LoRA is UNUSED at eval.** → **split fused → per-expert PEFT keys before submitting.**

This is likely why "attn+shared only" submissions cap ~0.56 and why the strong reference adapters
explicitly split expert keys. **Unlocking the routed experts is the biggest lever here.**

## What this notebook does
- Train: attn (q/k/v/o) + `in_proj` + MoE up/down (+lm_head), **NO out_proj**. requires_grad audit.
- **Weighted loss**: `W_CE·CE + W_DFT·DFT + W_HARD·maxmin` — DFT (reward-rectified) + worst-token
  mining (maximize min-logprob) + plain CE, all on the masked answer tokens. (Note: DFT down-weights
  hard tokens, maxmin up-weights them — they pull opposite ways; tune the weights.)
- **Diagnostic + repackage**: print every adapter key+shape, strip dead `out_proj`, **split fused
  routed-expert LoRA into per-expert keys**, fix namespace, verify against the expected native names,
  zip. RUN THE DIAGNOSTIC FIRST and check the printed format matches the splitter's assumptions.

> If the diagnostic shows an expert-key layout the splitter can't auto-handle, it keeps them + warns
> loudly — paste the printed shapes and the exact split can be written.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096
DROP_OVERLONG = True   # drop rows whose full seq > TRAIN_MAX_LEN (needed by the reused tokenize cell)
USE_FLASH_ATTN = 1
SEED = 42

def _find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    return ""
SFT_DATA_PATH = _find("/kaggle/input/**/balanced_sft.csv",
                      "/kaggle/input/**/merged_tong_andyhard.csv",
                      r"F:/Hackathons/Kaggle-Nemotron/data_manipulation/balanced_sft.csv")
WARM_START_ADAPTER_DIR = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"
OUT_ROOT = "outputs"; os.makedirs(OUT_ROOT, exist_ok=True)
SFT_ADAPTER_DIR = os.path.join(OUT_ROOT, "v32_adapter")        # raw Unsloth save
SUB_ADAPTER_DIR = os.path.join(OUT_ROOT, "v32_submittable")    # after expert-split repackage

# ── data knobs (consumed by the reused v24 data/tokenize cells) ──
SMOKE_TEST = 1
SMOKE_ROWS = 256
SUBSET_N = None
NUM_EPOCHS = 1
STRATIFIED_BATCHING = True
SYSTEM_PROMPT_MODE = "small"        # small|full085|none  (small recommended -> shorter seq)
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# ── MoE facts ──
N_EXPERTS = 128; RANK = 32

# ── LoRA (NO out_proj — dead on Unsloth) ──
ATTN_RANK = 32; LORA_ALPHA = 64
WARM_START = 1                       # 1: continue the 0.85 adapter; 0: fresh broad LoRA

# ── weighted loss: W_CE*CE + W_DFT*DFT + W_HARD*maxmin (worst-token) ──
W_CE = 1.0
W_DFT = 0.3                          # reward-rectified (down-weights low-prob teacher tokens)
W_HARD = 0.3                         # maximize min-logprob (push the worst answer tokens)
TOPK_MIN = 8                         # how many worst tokens per batch the maxmin term uses
WARMUP_MEAN_FRAC = 0.5               # train pure CE first half, then add DFT+maxmin (stability)

# ── optimizer (gentle refine on 0.85) ──
LEARNING_RATE = 1e-5; LR_SCHED = "cosine"; WARMUP_RATIO = 0.05
PER_DEV_BATCH = 1; GRAD_ACCUM = 16; MAX_GRAD_NORM = 1.0; WEIGHT_DECAY = 0.0

# ── repackage knobs (the expert-key unlock) ──
DROP_OUT_PROJ = True                 # strip the dead out_proj LoRA keys
SPLIT_FUSED_EXPERTS = True           # fused routed-expert LoRA -> per-expert PEFT keys
NAMESPACE = "auto"                   # auto|backbone|model  (model.model -> backbone for the eval)

print({"data": os.path.basename(SFT_DATA_PATH) or "(none)", "warm": WARM_START,
       "W_CE": W_CE, "W_DFT": W_DFT, "W_HARD": W_HARD, "split_experts": SPLIT_FUSED_EXPERTS,
       "SMOKE": SMOKE_TEST})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    _attn = "flash_attention_2" if USE_FLASH_ATTN else "eager"

    def _load(attn):
        return FastLanguageModel.from_pretrained(
            model_name=MODEL_PATH,
            max_seq_length=MODEL_MAX_LEN,
            load_in_4bit=False, load_in_8bit=False,
            full_finetuning=False,
            trust_remote_code=True,
            unsloth_force_compile=False,
            attn_implementation=attn,
            dtype=torch.bfloat16,
        )

    try:
        model, tokenizer = _load(_attn)
        print(f"Loaded with attn_implementation={_attn!r}")
    except Exception as e:
        print(f"[attn] {_attn} failed ({type(e).__name__}: {e}); falling back to eager")
        model, tokenizer = _load("eager")

    # Report the kernel actually in use. Per the forum (Benni): the native
    # modeling_nemotron_h.py loaded via trust_remote_code=True may leave FA2 OFF
    # even when requested -- the transformers-native impl (trust_remote_code=False)
    # is the one that enables FA2 + packed experts. Verify before trusting speed.
    try:
        _impl = getattr(model.config, "_attn_implementation", "?")
        print(f"[attn] effective config._attn_implementation = {_impl}")
    except Exception:
        pass

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"     # SFT loss wants right padding
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
# ── LoRA: attn + in_proj + MoE up/down (+lm_head). NO out_proj (dead on Unsloth). + audit ──
from unsloth import FastLanguageModel
from peft import PeftModel
import os, glob

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "up_proj", "down_proj", "lm_head"]

def _resolve(d):
    if d and os.path.exists(os.path.join(d, "adapter_config.json")): return d
    h = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return os.path.dirname(sorted(h, key=len)[0]) if h else None
A = _resolve(WARM_START_ADAPTER_DIR) if WARM_START else None

if A:
    print("[warm-start] continuing 0.85 adapter from", A,
          "(note: it may carry dead out_proj keys -> stripped at repackage)")
    model = PeftModel.from_pretrained(model, A, is_trainable=True)
    try: model.gradient_checkpointing_enable()
    except Exception as e: print("grad-ckpt warn:", e)
else:
    print("[fresh] broad LoRA (no out_proj):", target_modules)
    model = FastLanguageModel.get_peft_model(
        model, r=ATTN_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
        target_modules=target_modules, bias="none",
        use_gradient_checkpointing="unsloth", random_state=SEED)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try: model.config.use_cache = False
except Exception: pass
model.train()
model.print_trainable_parameters()

# ── FINDING 2: requires_grad audit (catch Unsloth's silent freeze of custom-named LoRA) ──
_lora = [(n, p) for n, p in model.named_parameters()
         if ("lora_A" in n or "lora_B" in n or "lora_magnitude" in n)]
_frozen = [n for n, p in _lora if not p.requires_grad]
if _frozen:
    for _, p in _lora: p.requires_grad_(True)
    print(f"[audit] re-enabled {len(_frozen)} silently-frozen LoRA params (finding 2 bug)")
_out = [n for n, _ in _lora if ".out_proj." in n]
_exp = [n for n, _ in _lora if ".experts." in n and ".shared_expert" not in n]
print(f"[audit] LoRA tensors={len(_lora)} | out_proj(dead)={len(_out)} | routed-expert={len(_exp)}")
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[audit] trainable={n_train/1e6:.1f}M")


In [ ]:
_FULL085 = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses.
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F carefully. State the source and target base. No prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units. Round only at the end.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme and direction. Transform \
one character at a time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations; or apply the defined transformation rule literally. Keep equations balanced.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

_SMALL = (
    "You solve a puzzle defined only by the examples in the prompt. Infer the exact "
    "rule from the examples, apply it to the query, and verify your result reproduces "
    "the examples before answering. Reason step by step, then output the final answer "
    "once as \\boxed{...} with nothing after it."
)

if SYSTEM_PROMPT_MODE == "full085":
    SYSTEM_PROMPT = _FULL085
elif SYSTEM_PROMPT_MODE == "small":
    SYSTEM_PROMPT = _SMALL
elif SYSTEM_PROMPT_MODE == "none":
    SYSTEM_PROMPT = ""   # NOTE: cell still emits a system message; empty content is fine for Nemotron's template
else:
    raise ValueError(f"bad SYSTEM_PROMPT_MODE={SYSTEM_PROMPT_MODE!r}")

print(f"SYSTEM_PROMPT mode={SYSTEM_PROMPT_MODE} chars={len(SYSTEM_PROMPT)}")


In [ ]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")

def _find(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n in low: return low[n]
    return None

PROMPT_COL = _find(df_sft.columns, ["prompt", "question", "problem", "input"])
ANSWER_COL = _find(df_sft.columns, ["answer", "solution", "label", "target", "final_answer"])
COT_COL    = _find(df_sft.columns, ["cot", "reasoning", "think", "generated_cot",
                                    "response", "completion", "rationale", "output"])
TYPE_COL   = _find(df_sft.columns, ["type", "category", "puzzle_type", "task_type"])
print(f"Detected -> prompt={PROMPT_COL!r}  answer={ANSWER_COL!r}  cot={COT_COL!r}  type={TYPE_COL!r}")
if PROMPT_COL is None:
    raise ValueError(f"No prompt-like column in {list(df_sft.columns)}")

df_sft = df_sft.dropna(subset=[PROMPT_COL]).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    df_sft = df_sft.head(SMOKE_ROWS).reset_index(drop=True)
    print(f"[SMOKE] using {len(df_sft)} rows (tiny dry-run; set SMOKE_TEST=0 to train on all)")
elif SUBSET_N is not None:
    df_sft = df_sft.head(SUBSET_N).reset_index(drop=True)
    print(f"[REAL] using subset of {len(df_sft)} rows (set SUBSET_N=None to train on all)")
else:
    print(f"[REAL] using ALL {len(df_sft)} rows")


def _strip_boxed(text):
    r"""Remove every \boxed{...} via BRACE-BALANCED matching.

    `re.sub(r'\\boxed\{[^{}]*\}', ...)` cannot match a box whose CONTENT has a
    brace (cipher answers are literally things like `(/&{`) -> the inline box
    survives -> the wrapper adds a 2nd box -> the old `count==1` check raised
    'malformed targets'. This walks braces so any inline box is fully removed.
    """
    tok = "\\boxed{"
    out, i = [], 0
    while i < len(text):
        j = text.find(tok, i)
        if j == -1:
            out.append(text[i:]); break
        out.append(text[i:j])
        k = j + len(tok); depth = 1
        while k < len(text) and depth > 0:
            if text[k] == "{": depth += 1
            elif text[k] == "}": depth -= 1
            k += 1
        i = k
    return "".join(out)


def build_assistant_text(row):
    r"""Canonical target: <think>\n{reasoning}\n</think>\n\boxed{ans}.

    ans is taken verbatim from the answer column (may contain braces/symbols).
    """
    ans = "" if ANSWER_COL is None else str(row[ANSWER_COL]).strip()
    cot = "" if COT_COL is None else str(row.get(COT_COL, "") or "").strip()
    cot = cot.replace("<think>", "").replace("</think>", "").strip()
    cot = cot.replace("\r\n", "\n").replace("\r", "\n")  # v24: normalize CRLF
    cot = re.sub(r'(?im)^.*I will (now )?(put|return) .*\\boxed\{\}.*$', '', cot)
    cot = re.sub(r'(?im)^.*The answer .*\\boxed.*$', '', cot)
    cot = _strip_boxed(cot)                      # brace-balanced inline-box removal
    cot = re.sub(r'\n{3,}', '\n\n', cot).strip()
    think = cot if cot else "Work through the problem step by step."
    return f"<think>\n{think}\n</think>\n\\boxed{{{ans}}}"

records, record_types = [], []
for _, row in df_sft.iterrows():
    records.append({
        "system":    SYSTEM_PROMPT,
        "user":      str(row[PROMPT_COL]) + PROMPT_SUFFIX,
        "assistant": build_assistant_text(row),
    })
    record_types.append(str(row[TYPE_COL]) if TYPE_COL else "unknown")

# Format check (presence-based, brace-tolerant): a closing </think> must exist and
# a \boxed{ must appear AFTER the last </think>. Do NOT count boxes or require a
# trailing '}' -- answers can legitimately contain braces.
def _well_formed(a):
    if "</think>" not in a:
        return False
    return "\\boxed{" in a.rsplit("</think>", 1)[-1]

_keep_r, _keep_t, _n_bad = [], [], 0
for _r, _t in zip(records, record_types):
    if _well_formed(_r["assistant"]):
        _keep_r.append(_r); _keep_t.append(_t)
    else:
        _n_bad += 1
records, record_types = _keep_r, _keep_t
print(f"[format] dropped {_n_bad} malformed; kept {len(records)} well-formed targets.")
if len(records) == 0:
    raise ValueError("All targets malformed -- check answer/cot columns of SFT_DATA_PATH.")

raw_ds = HFDataset.from_list(records)
print("Type distribution:", dict(pd.Series(record_types).value_counts().head(10).to_dict()))
print("\n--- sample target TAIL ---\n", records[0]["assistant"][-160:])

In [ ]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)
    full_text   = render(full_msgs,   False)
    prefix_text = render(prefix_msgs, True)
    # v24: tokenize WITHOUT truncation so we can DROP overlong rows (truncation would
    # silently cut the trailing \boxed{} -> a boxless target that teaches nothing).
    full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels, "n_tok": len(full_ids)}

tokenized_ds = raw_ds.map(tokenize_with_assistant_mask,
                          remove_columns=raw_ds.column_names,
                          desc="Tokenize + assistant mask")

# v24: drop fully-masked AND overlong rows; report per-category survivors.
from collections import Counter
_kept_rows, _kept_types, _n_all_masked, _n_overlong = [], [], 0, 0
for i, ex in enumerate(tokenized_ds):
    if not any(t != -100 for t in ex["labels"]):
        _n_all_masked += 1; continue
    if DROP_OVERLONG and ex["n_tok"] > TRAIN_MAX_LEN:
        _n_overlong += 1; continue
    ids, lab = ex["input_ids"], ex["labels"]
    if ex["n_tok"] > TRAIN_MAX_LEN:                 # DROP_OVERLONG off -> hard truncate
        ids, lab = ids[:TRAIN_MAX_LEN], lab[:TRAIN_MAX_LEN]
    _kept_rows.append({"input_ids": ids, "labels": lab})
    _kept_types.append(record_types[i])
tokenized_ds = HFDataset.from_list(_kept_rows)
record_types = _kept_types
print(f"Kept {len(tokenized_ds)} rows (dropped {_n_all_masked} fully-masked, "
      f"{_n_overlong} overlong >{TRAIN_MAX_LEN}).")
print("per-category kept:", dict(Counter(record_types)))

if len(tokenized_ds) == 0:
    raise RuntimeError("tokenized_ds EMPTY -> every row fully masked. Masking broken.")

_ex0 = tokenized_ds[0]
_un = [t for t, l in zip(_ex0["input_ids"], _ex0["labels"]) if l != -100]
print(f"[mask-check] row0 total={len(_ex0['input_ids'])} unmasked={len(_un)}")
print("[mask-check] decoded unmasked:", repr(tokenizer.decode(_un)[:200]))
assert "boxed" in tokenizer.decode(_un), "[mask-check] no boxed in unmasked span -> misaligned."

import numpy as np
_lens = np.array([len(x["input_ids"]) for x in tokenized_ds])
_ul   = np.array([sum(1 for l in x["labels"] if l != -100) for x in tokenized_ds])
print(f"len min={_lens.min()} mean={_lens.mean():.0f} p90={int(np.percentile(_lens,90))} max={_lens.max()}")
print(f"unmasked/seq min={_ul.min()} mean={_ul.mean():.0f} max={_ul.max()}")
print(f"[v24] total training tokens = {int(_lens.sum()):,}  (runtime scales ~linearly with this)")


In [ ]:
# ── weighted-loss SFT: W_CE*CE + W_DFT*DFT + W_HARD*maxmin (worst-token / max-min-logprob) ──
import os, time, gc, torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
os.environ["TORCHDYNAMO_DISABLE"] = "1"; os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

class Collate:
    def __init__(self, tok): self.pad = tok.pad_token_id
    def __call__(self, fs):
        m = max(len(f["input_ids"]) for f in fs); ii = []; lb = []; am = []
        for f in fs:
            ids = list(f["input_ids"]); la = list(f["labels"]); p = m - len(ids)
            ii.append(ids + [self.pad] * p); lb.append(la + [-100] * p); am.append([1] * len(ids) + [0] * p)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(lb)}
collator = Collate(tokenizer)

class WLossTrainer(Trainer):
    def __init__(self, *a, warmup_mean_steps=0, **k):
        super().__init__(*a, **k); self.warm = warmup_mean_steps; self._dbg = 0
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs); logits = out.logits
        sl = logits[:, :-1, :]; slb = labels[:, 1:].to(sl.device); mask = (slb != -100)
        V = sl.shape[-1]; fm = mask.reshape(-1)
        sel = sl.reshape(-1, V)[fm].float(); selab = slb.reshape(-1)[fm]
        if selab.numel() == 0:
            loss = (logits.float().sum() * 0.0).requires_grad_(True)
            return (loss, out) if return_outputs else loss
        lp = F.log_softmax(sel, dim=-1)
        nll = (-lp.gather(-1, selab[:, None]).squeeze(-1))
        nll = torch.nan_to_num(nll, nan=0.0, posinf=30.0, neginf=0.0)
        ce = nll.mean()
        if self.state.global_step < self.warm:
            loss = ce                                            # pure-CE warmup (stability)
            comp = "CE"
        else:
            p_gold = nll.detach().neg().exp()
            dft = (p_gold * nll).mean()                          # DFT (reward-rectified)
            k = min(TOPK_MIN, nll.numel())
            hard = torch.topk(nll, k).values.mean()              # maximize min-logprob (worst tokens)
            loss = W_CE * ce + W_DFT * dft + W_HARD * hard
            comp = f"{W_CE}*CE+{W_DFT}*DFT+{W_HARD}*maxmin"
        if self._dbg < 3:
            self._dbg += 1
            print(f"[loss step~{self.state.global_step}] ce={float(ce):.3f} loss={float(loss):.3f} "
                  f"ntok={int(mask.sum())} ({comp})")
        if not torch.isfinite(loss): loss = ce
        return (loss, out) if return_outputs else loss

_eff = PER_DEV_BATCH * GRAD_ACCUM
_total = max(1, len(tokenized_ds) // _eff * NUM_EPOCHS)
_warm = max(1, int(WARMUP_RATIO * _total))
_warm_mean = int(WARMUP_MEAN_FRAC * (8 if SMOKE_TEST else _total))
args = TrainingArguments(
    output_dir=os.path.join(OUT_ROOT, "run"), num_train_epochs=NUM_EPOCHS,
    max_steps=8 if SMOKE_TEST else -1, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHED,
    warmup_steps=_warm, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM, optim="paged_adamw_8bit",
    bf16=True, gradient_checkpointing=False, remove_unused_columns=False, logging_steps=1,
    report_to="none", save_strategy="no", seed=SEED)
print(f"steps~{_total} warmup={_warm} pure-CE-first={_warm_mean} steps | loss=W_CE*CE+W_DFT*DFT+W_HARD*maxmin")

trainer = WLossTrainer(model=model, args=args, train_dataset=tokenized_ds, data_collator=collator,
                       warmup_mean_steps=_warm_mean)
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.train()
print(f"train done {(time.time()-t0)/60:.1f} min | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

os.makedirs(SFT_ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(SFT_ADAPTER_DIR); tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print("raw adapter saved ->", SFT_ADAPTER_DIR, "(repackage next for submittable expert keys)")


In [ ]:
# ── DIAGNOSTIC + REPACKAGE: strip dead out_proj, split fused routed-experts -> per-expert PEFT keys ──
import os, re, json, zipfile, collections, torch
from safetensors import safe_open
from safetensors.torch import save_file

SRC = SFT_ADAPTER_DIR; DST = SUB_ADAPTER_DIR; os.makedirs(DST, exist_ok=True)
T = {}
with safe_open(os.path.join(SRC, "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    for k in f.keys():
        T[k] = f.get_tensor(k)
src_cfg = json.load(open(os.path.join(SRC, "adapter_config.json")))
print(f"=== adapter: {len(T)} tensors, r={src_cfg.get('r')} ===")

# ---- DIAGNOSTIC: leaf-module groups + example shapes (run this FIRST, eyeball it) ----
grp = collections.defaultdict(list)
for k, v in T.items():
    base = re.sub(r"\.lora_[AB]\.weight$", "", re.sub(r"\.weight$", "", k))
    leaf = base.split(".")[-1]
    grp[leaf].append((k, tuple(v.shape)))
print("--- key groups (leaf: count | example tail | shape) ---")
for leaf in sorted(grp):
    k0, s0 = grp[leaf][0]
    print(f"  {leaf:18s} n={len(grp[leaf]):5d} | ...{'.'.join(k0.split('.')[-4:])} | {s0}")

def is_routed_fused(k):
    return (".experts." in k) and (".shared_expert" not in k) and not re.search(r"\.experts\.\d+\.", k)
fused = [k for k in T if is_routed_fused(k)]
n_outproj = sum(".out_proj." in k for k in T)
print(f"\nrouted-expert FUSED keys: {len(fused)} | dead out_proj keys: {n_outproj}")

# ---- namespace (eval native model uses backbone.* not model.model.*) ----
ns = NAMESPACE
if ns == "auto":
    ns = "backbone" if any("base_model.model.model." in k for k in T) else "keep"
print("namespace mode:", ns)
def rn(k):
    return k.replace("base_model.model.model.", "base_model.model.backbone.") if ns == "backbone" else k

# ---- build submittable tensors ----
out = {}; n_split = n_drop = n_keep = warn = 0
for k, v in T.items():
    if DROP_OUT_PROJ and ".out_proj." in k:
        n_drop += 1; continue
    if SPLIT_FUSED_EXPERTS and is_routed_fused(k):
        AB = "lora_A" if "lora_A" in k else ("lora_B" if "lora_B" in k else None)
        kl = k.lower()
        proj = "up_proj" if ("up" in kl or "gate_up" in kl) else ("down_proj" if "down" in kl else None)
        per = None
        if v.dim() == 3 and v.shape[0] == N_EXPERTS:
            per = [v[j].contiguous() for j in range(N_EXPERTS)]
        elif v.dim() == 2 and v.shape[0] % N_EXPERTS == 0:
            per = [t.contiguous() for t in v.reshape(N_EXPERTS, -1, v.shape[1])]
        elif v.dim() == 2 and v.shape[1] % N_EXPERTS == 0:
            per = [t.contiguous() for t in v.reshape(v.shape[0], N_EXPERTS, -1).permute(1, 0, 2)]
        if per is None or AB is None or proj is None:
            out[rn(k)] = v; warn += 1
            print(f"  [keep-UNSPLITTABLE] {k} {tuple(v.shape)} (AB={AB} proj={proj}) -> paste this shape")
            continue
        prefix = re.sub(r"\.experts\..*", "", k)          # ...mixer.experts
        for j, t in enumerate(per):
            out[rn(f"{prefix}.experts.{j}.{proj}.{AB}.weight")] = t.contiguous()
        n_split += 1
    else:
        out[rn(k)] = v; n_keep += 1

print(f"\nrepackage: split {n_split} fused -> {n_split*N_EXPERTS} per-expert keys | "
      f"dropped {n_drop} out_proj | kept {n_keep} | unsplittable {warn}")
print("total submittable keys:", len(out))

# ---- config from final keys ----
cfg = dict(src_cfg)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME; cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0; cfg["modules_to_save"] = None
cfg["target_modules"] = sorted({re.sub(r"\.lora_[AB]\.weight$", "", k).split(".")[-1] for k in out})
json.dump(cfg, open(os.path.join(DST, "adapter_config.json"), "w"), indent=2)
save_file(out, os.path.join(DST, "adapter_model.safetensors"))

# ---- verify ----
pe = [k for k in out if re.search(r"\.experts\.\d+\.(up|down)_proj\.lora_[AB]\.weight", k)]
print(f"[verify] per-expert keys: {len(pe)} | target_modules: {cfg['target_modules']}")
print(f"[verify] sample expert key: {pe[0] if pe else '(NONE -> experts not unlocked!)'}")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUT_ROOT
z = os.path.join(WORK, "submission.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for n in ("adapter_config.json", "adapter_model.safetensors"):
        zf.write(os.path.join(DST, n), n)
print(f"submission.zip -> {z} ({os.path.getsize(z)/1e6:.0f} MB)")
print("\nNOTE: if 'routed-expert FUSED keys'=0 AND 'per-expert keys'=0, Unsloth never saved routed-expert")
print("LoRA in PEFT form -> the 856M experts were untrained/unsaved. If 'UNSPLITTABLE' warned, paste the")
print("shapes. Otherwise this zip now carries the routed experts the evaluator can actually load.")
